<a href="https://colab.research.google.com/github/jeevithajeevitha3420-create/Self-Correcting-Study-Planner-langgraph/blob/main/Self_Correcting_Study_Planner_langgraph_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langgraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class StudyState(TypedDict):
    subjects: list
    hours: int
    days: int
    plan: list
    status: str
    resources: dict
    feedback: str
    weak_topics: list


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def create_plan(state):

    subjects = state["subjects"]
    hours = state["hours"]
    days = state["days"]

    plan = []

    daily_hours = hours
    subject_count = len(subjects)

    for day in range(1, days + 1):

        day_plan = []

        for subject in subjects:

            time = round(daily_hours / subject_count, 1)

            day_plan.append(
                f"{subject} - {time} hour"
            )

        plan.append({
            "day": day,
            "tasks": day_plan
        })

    return {
        "plan": plan
    }

In [ ]:
builder = StateGraph(StudyState)

builder.add_node("create_plan", create_plan)
builder.add_node("check_workload", check_workload)
builder.add_node("correct_plan", correct_plan)

builder.add_edge(START, "create_plan")

builder.add_edge(
    "create_plan",
    "check_workload"
)

builder.add_conditional_edges(
    "check_workload",
    decide_next,
    {
        "correct": "correct_plan",
        "finish": END
    }
)

builder.add_edge(
    "correct_plan",
    "check_workload"
)

graph = builder.compile()

In [ ]:
result = graph.invoke({
    "subjects": ["Python", "DBMS", "Machine Learning"],
    "hours": 3,
    "days": 3,
    "plan": []
})

for day in result["plan"]:
    print(f"\nDay {day['day']}")

    for task in day["tasks"]:
        print("  ", task)


Day 1
   Python - 1.0 hour
   DBMS - 1.0 hour
   Machine Learning - 1.0 hour

Day 2
   Python - 1.0 hour
   DBMS - 1.0 hour
   Machine Learning - 1.0 hour

Day 3
   Python - 1.0 hour
   DBMS - 1.0 hour
   Machine Learning - 1.0 hour


In [ ]:
def check_workload(state):

    total_hours = 0

    for day in state["plan"]:
        for task in day["tasks"]:
            time = float(task.split("-")[1].replace("hour", "").strip())
            total_hours += time

    available_hours = state["hours"] * state["days"]

    if total_hours <= available_hours:
        status = "realistic"
    else:
        status = "too_much"

    return {
        "status": status
    }

In [ ]:
def correct_plan(state):

    print("⚠️ Plan is too heavy. Correcting...")

    new_plan = []

    for day in state["plan"]:

        tasks = []

        for task in day["tasks"]:

            subject = task.split("-")[0].strip()

            tasks.append(
                f"{subject} - 0.5 hour"
            )

        new_plan.append({
            "day": day["day"],
            "tasks": tasks
        })

    return {
        "plan": new_plan
    }

In [ ]:
def decide_next(state):

    if state["status"] == "realistic":
        return "finish"

    return "correct"

In [ ]:
result = graph.invoke({
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [],
    "status": ""
})

print("\nFINAL PLAN\n")

for day in result["plan"]:

    print(f"\nDay {day['day']}")

    for task in day["tasks"]:
        print(" ", task)


FINAL PLAN


Day 1
  Python - 1.0 hour
  DBMS - 1.0 hour
  Machine Learning - 1.0 hour

Day 2
  Python - 1.0 hour
  DBMS - 1.0 hour
  Machine Learning - 1.0 hour

Day 3
  Python - 1.0 hour
  DBMS - 1.0 hour
  Machine Learning - 1.0 hour


In [ ]:
import requests
import google.auth

print("requests:", requests.__version__)
print("google-auth:", google.auth.__version__)

requests: 2.32.4
google-auth: 2.58.0


In [ ]:
import requests

url = "https://www.google.com/search?q=Python+tutorial"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

print(response.status_code)

200


In [ ]:
from urllib.parse import quote


def search_web(topic):

    query = quote(topic)

    url = f"https://www.google.com/search?q={query}"

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    parser = LinkParser()
    parser.feed(response.text)

    return parser.links[:5]

In [ ]:
html = search_web("Python machine learning tutorial")

print(html[:1000])

<!DOCTYPE html><html lang="zh-TW"><head><title>Google Search</title><style>body{background-color:#fff}</style><script nonce="4HJyduWUyej81OYSmZuIIw">window.google = window.google || {};window.google.c = window.google.c || {cap:0};</script></head><body><noscript><style>table,div,span,p{display:none}</style><meta content="0;url=/httpservice/retry/enablejs?sei=7KS0asiyJ7Ld1e8PoPXf0Qo" http-equiv="refresh"><div style="display:block">如果系統沒有在數秒鐘後將您重新導向，請按一下<a href="/httpservice/retry/enablejs?sei=7KS0asiyJ7Ld1e8PoPXf0Qo">這裡</a>。</div></noscript><script nonce="4HJyduWUyej81OYSmZuIIw">(function(){var sctm=false;(function(){sctm&&google.tick("load","pbsst");}).call(this);})();</script><script nonce="4HJyduWUyej81OYSmZuIIw">//# sourceMappingURL=data:application/json;charset=utf-8;base64,eyJ2ZXJzaW9uIjogMywic291cmNlcyI6WyIiXSwic291cmNlc0NvbnRlbnQiOlsiICJdLCJuYW1lcyI6WyJjbG9zdXJlRHluYW1pY0J1dHRvbiJdLCJtYXBwaW5ncyI6IkFBQUE7QUFBQTtBQUFBO0FBQUE7QUFBQTtBQUFBO0FBQUEifQ==
(function(){var l=function(p,Z,

In [ ]:
from html.parser import HTMLParser
from urllib.parse import quote
import requests

In [ ]:
class LinkParser(HTMLParser):

    def __init__(self):
        super().__init__()

        self.links = []
        self.current_link = None
        self.current_text = ""

    def handle_starttag(self, tag, attrs):

        if tag == "a":

            for key, value in attrs:

                if key == "href" and value.startswith("http"):

                    self.current_link = value
                    self.current_text = ""

    def handle_data(self, data):

        if self.current_link:
            self.current_text += data

    def handle_endtag(self, tag):

        if tag == "a" and self.current_link:

            title = self.current_text.strip()

            if title:

                self.links.append({
                    "title": title,
                    "url": self.current_link
                })

            self.current_link = None
            self.current_text = ""

In [ ]:
def search_web(topic):

    query = quote(topic)

    url = f"https://www.google.com/search?q={query}"

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    parser = LinkParser()
    parser.feed(response.text)

    return parser.links[:5]

In [ ]:
results = search_web("Python beginner tutorial")

for link in results:
    print(link)

https://support.google.com/websearch


In [ ]:
from typing import TypedDict

In [ ]:
class StudyState(TypedDict):

    subjects: list
    hours: int
    days: int
    plan: list
    status: str
    resources: dict

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import requests
from urllib.parse import quote
from html.parser import HTMLParser

In [ ]:
class LinkParser(HTMLParser):

    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):

        if tag == "a":

            for key, value in attrs:

                if key == "href" and value.startswith("http"):
                    self.links.append(value)


def search_web(topic):

    query = quote(topic)

    url = f"https://www.google.com/search?q={query}"

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    parser = LinkParser()
    parser.feed(response.text)

    return parser.links[:5]

In [ ]:
from ddgs import DDGS

def search_resources(state):
    resources = {}

    topics = state["weak_topics"]

    for subject in topics:
        print(f"🔎 Searching resources for {subject}...")

        results = DDGS().text(
            f"{subject} beginner tutorial practice problems",
            max_results=3
        )

        resources[subject] = results

    return {
        "resources": resources
    }

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
builder = StateGraph(StudyState)

builder.add_node(
    "search_resources",
    search_resources
)

builder.add_edge(
    START,
    "search_resources"
)

builder.add_edge(
    "search_resources",
    END
)

graph = builder.compile()

In [ ]:
result = graph.invoke({
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [],
    "status": "",
    "resources": {}
})

Searching for Python...
Searching for DBMS...
Searching for Machine Learning...


In [ ]:
for subject, links in result["resources"].items():

    print("\n" + "=" * 40)
    print(subject)
    print("=" * 40)

    for link in links:
        print(link)


Python
https://support.google.com/websearch

DBMS
https://support.google.com/websearch

Machine Learning
https://support.google.com/websearch


In [ ]:
results = search_web("Python beginner tutorial")

for result in results:

    print("Title:", result["title"])
    print("URL:", result["url"])
    print()

Title: 意見回饋
URL: https://support.google.com/websearch



In [ ]:
!pip install -q duckduckgo-search

In [ ]:
from duckduckgo_search import DDGS

results = DDGS().text(
    "Python beginner tutorial",
    max_results=5
)

for result in results:
    print("Title:", result["title"])
    print("URL:", result["href"])
    print("Description:", result["body"])
    print()

/tmp/ipykernel_5835/2416327121.py:3: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use ti

In [ ]:
!pip install -q ddgs

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
from ddgs import DDGS

results = DDGS().text(
    "Python beginner tutorial",
    max_results=5
)

for result in results:
    print("Title:", result["title"])
    print("URL:", result["href"])
    print("Description:", result["body"])
    print()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Title: Go from Beginner to Python Pro - Enter the World of Python
URL: https://www.bing.com/aclick?ld=e8_Mqpdc6PgXp_o8gYoOQRQzVUCUwUQ-0xQyGjMf5GYf3NVet3b3awpymJOwIk1gtEX-tg3OL3jYIuB94m_L42-UjSstwI5H6pYNHyJNZ-MJFQ6p0MJYMdoORkXxPb9Ux5Sqpv4LSc3u62bhPQrmwJut9sDqghEMYaYxVZwqsRG1TNVB_vg7z3vWgiOc9IYmdrWqW65wAmSiROT96QkfqLJh5X6Bw&u=aHR0cHMlM2ElMmYlMmZ3d3cuY291cnNlcmEub3JnJTJmcHJvZmVzc2lvbmFsLWNlcnRpZmljYXRlcyUyZm1pY3Jvc29mdC1weXRob24tZGV2ZWxvcGVyJTNmdXRtX21lZGl1bSUzZHNlbSUyNnV0bV9zb3VyY2UlM2RiZyUyNnV0bV9jYW1wYWlnbiUzZGIyY19hcGFjX21pY3Jvc29mdC1weXRob24tZGV2ZWxvcGVyX21pY3Jvc29mdF9mdGNvZl9wcm9mZXNzaW9uYWwtY2VydGlmaWNhdGVzX2N4X2RyX2JhdV9iZ19zZW1fcHItcGhfczFfYWxsX21feF8yNS0wMl94JTI2Y2FtcGFpZ25pZCUzZDY2MzM1ODIxOSUyNmFkZ3JvdXBpZCUzZDEyNTIzNDUwODY5MTYxODUlMjZkZXZpY2UlM2RjJTI2a2V5d29yZCUzZGludHJvZHVjdGlvbiUyNTIwcHl0aG9uJTI1MjBwcm9ncmFtbWluZyUyNm1hdGNodHlwZSUzZGUlMjZuZXR3b3JrJTNkbyUyNmRldmljZW1vZGVsJTNkJTI2YWRwb3NpdGlvbiUzZCUyNmNyZWF0aXZlaWQlM2Q3ODI3MTY5OTA2NjcxMSUyNmFkZ3JvdXAlM2RiMmNfYXBhY19taWNyb3NvZn

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def search_resources(state):

    resources = {}

    for subject in state["subjects"]:

        print(f"🔎 Searching: {subject}")

        results = DDGS().text(
            f"{subject} beginner tutorial documentation",
            max_results=3
        )

        resources[subject] = results

    return {
        "resources": resources
    }


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
result = search_resources({
    "subjects": ["Python", "DBMS", "Machine Learning"]
})

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching: Python


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching: DBMS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching: Machine Learning


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
for subject, results in result["resources"].items():

    print("\n====================")
    print(subject)
    print("====================")

    for item in results:

        print("📌", item["title"])
        print("🔗", item["href"])
        print("📝", item["body"][:200])
        print()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Python
📌 The Python Tutorial — Python 3.14.7 documentation
🔗 https://docs.python.org/3/tutorial/index.html
📝 1 day ago · This tutorial introduces the reader informally to the basic concepts and features of the Python language and system. Be aware that it expects you to have a basic understanding of programmi

📌 Python Tutorial - W3SchoolsUsage example
🔗 https://www.w3schools.com/python/
📝 Learn Python in bite-sized quests with XP and streaks. Your progress grows for each new adventure! Free on iOS and Android. With our "Try it Yourself" editor, you can edit Python code and view the res

📌 Python For Beginners | Python.org
🔗 https://www.python.org/about/gettingstarted/
📝 The online documentation is your first port of call for definitive information. There is a fairly brief tutorial that gives you basic information about the language and gets you started.


DBMS
📌 DBMS Tutorial
🔗 https://www.tutorialspoint.com/dbms/index.htm
📝 This tutorial explains the basics of DBMS such as its archit

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
from typing import TypedDict

class StudyState(TypedDict):
    subjects: list
    hours: int
    days: int
    plan: list
    status: str
    resources: dict

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def create_plan(state):

    subjects = state["subjects"]
    hours = state["hours"]
    days = state["days"]

    time_per_subject = round(hours / len(subjects), 1)

    plan = []

    for day in range(1, days + 1):

        tasks = []

        for subject in subjects:

            tasks.append({
                "subject": subject,
                "hours": time_per_subject
            })

        plan.append({
            "day": day,
            "tasks": tasks
        })

    return {
        "plan": plan
    }

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def check_workload(state):

    total_planned = 0

    for day in state["plan"]:

        for task in day["tasks"]:

            total_planned += task["hours"]

    available = state["hours"] * state["days"]

    if total_planned <= available:

        return {
            "status": "realistic"
        }

    else:

        return {
            "status": "too_much"
        }

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
from ddgs import DDGS

def search_resources(state):

    resources = {}

    for subject in state["subjects"]:

        print(f"🔎 Searching web for {subject}...")

        results = DDGS().text(
            f"{subject} beginner tutorial documentation",
            max_results=3
        )

        resources[subject] = results

    return {
        "resources": resources
    }

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def decide_next(state):

    if state["status"] == "realistic":

        return "search"

    return "correct"

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def correct_plan(state):

    print("🔄 Correcting study plan...")

    new_plan = []

    for day in state["plan"]:

        tasks = []

        for task in day["tasks"]:

            tasks.append({
                "subject": task["subject"],
                "hours": 0.5
            })

        new_plan.append({
            "day": day["day"],
            "tasks": tasks
        })

    return {
        "plan": new_plan
    }

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(StudyState)

builder.add_node("create_plan", create_plan)

builder.add_node(
    "check_workload",
    check_workload
)

builder.add_node(
    "correct_plan",
    correct_plan
)

builder.add_node(
    "search_resources",
    search_resources
)

builder.add_edge(
    START,
    "create_plan"
)

builder.add_edge(
    "create_plan",
    "check_workload"
)

builder.add_conditional_edges(
    "check_workload",
    decide_next,
    {
        "correct": "correct_plan",
        "search": "search_resources"
    }
)

builder.add_edge(
    "correct_plan",
    "check_workload"
)

builder.add_edge(
    "search_resources",
    END
)

graph = builder.compile()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
result = graph.invoke({

    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],

    "hours": 3,

    "days": 3,

    "plan": [],

    "status": "",

    "resources": {}
})

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Python...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for DBMS...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Machine Learning...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
for subject, results in result["resources"].items():

    print("\n" + "=" * 50)
    print(subject)
    print("=" * 50)

    for item in results:

        print("📌", item["title"])
        print("🔗", item["href"])
        print("📝", item["body"][:150])
        print()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Python
📌 The Python Tutorial — Python 3.14.7 documentation
🔗 https://docs.python.org/3/tutorial/index.html
📝 The Python interpreter is easily ... customizable applications. This tutorial introduces the reader informally to the basic concepts and features of t

📌 #15 Python Documentation- Python Tutorials For Beginners - YouTube
🔗 https://www.youtube.com/watch?v=FvBslBkZIO4
📝 Python document link- https://docs.python.org/3/Python Serieshttps://www.youtube.com/playlist?list=PL6flErFppaj0v1XGN-og_hL4Rw2v2lidKAll Playlisthttps

📌 Our Documentation | Python.org
🔗 https://www.python.org/doc/
📝 Python's documentation, tutorials, and guides are constantly evolving. Get started here, or scroll down for documentation broken out by type and subje


DBMS
📌 DBMS Tutorial - GeeksforGeeks
🔗 https://www.geeksforgeeks.org/dbms/dbms/
📝 Mar 14, 2026 · Understand the basics of DBMS, explore how it works, and learn about different database architectures used in real-world systems. It is

📌 DBMS TutorialGe

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def analyze_feedback(state):
    feedback = state["feedback"].lower()

    new_plan = []

    for day in state["plan"]:
        tasks = []

        for task in day["tasks"]:

            subject = task["subject"]
            hours = task["hours"]

            if subject.lower() in feedback:

                if any(word in feedback for word in [
                    "difficult",
                    "struggle",
                    "couldn't",
                    "did not",
                    "not understand"
                ]):
                    hours += 0.5

            tasks.append({
                "subject": subject,
                "hours": hours
            })

        new_plan.append({
            "day": day["day"],
            "tasks": tasks
        })

    print("🔄 Plan adjusted using student feedback.")

    return {
        "plan": new_plan
    }

In [ ]:
builder.add_node(
    "analyze_feedback",
    analyze_feedback
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
builder.add_edge(
    "search_resources",
    "analyze_feedback"
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
builder.add_edge(
    "analyze_feedback",
    END
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
result = graph.invoke({

    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],

    "hours": 3,

    "days": 3,

    "plan": [],

    "status": "",

    "resources": {},

    "feedback": """
    I completed Python.
    I completed only half of DBMS.
    I did not study Machine Learning.
    """
})

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Python...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for DBMS...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Machine Learning...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
result = graph.invoke({

    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],

    "hours": 3,
    "days": 3,

    "plan": [],

    "status": "",

    "resources": {},

    "feedback": """
    I completed Python.
    I completed only half of DBMS.
    I did not study Machine Learning.
    """
})

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Python...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for DBMS...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔎 Searching web for Machine Learning...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
for day in result["plan"]:

    print(f"\n📅 Day {day['day']}")

    for task in day["tasks"]:

        print(
            f"   {task['subject']}: "
            f"{task['hours']} hour"
        )


📅 Day 1
   Python: 1.0 hour
   DBMS: 1.0 hour
   Machine Learning: 1.0 hour

📅 Day 2
   Python: 1.0 hour
   DBMS: 1.0 hour
   Machine Learning: 1.0 hour

📅 Day 3
   Python: 1.0 hour
   DBMS: 1.0 hour
   Machine Learning: 1.0 hour


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
def extract_weak_topics(state):
    feedback = state["feedback"].lower()

    weak_topics = []

    for subject in state["subjects"]:
        subject_name = subject.lower()

        if subject_name in feedback:
            if any(word in feedback for word in [
                "difficult",
                "struggle",
                "couldn't",
                "could not",
                "did not",
                "not complete",
                "unfinished",
                "not understand",
                "half"
            ]):
                weak_topics.append(subject)

    print("📌 Weak topics:", weak_topics)

    return {
        "weak_topics": weak_topics
    }


In [ ]:
builder.add_node("extract_weak_topics", extract_weak_topics)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
builder.add_edge("analyze_feedback", "extract_weak_topics")
builder.add_edge("extract_weak_topics", "search_resources")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(StudyState)

builder.add_node("create_plan", create_plan)
builder.add_node("check_workload", check_workload)
builder.add_node("correct_plan", correct_plan)
builder.add_node("search_resources", search_resources)
builder.add_node("analyze_feedback", analyze_feedback)
builder.add_node("extract_weak_topics", extract_weak_topics)

In [ ]:
builder.add_edge(START, "create_plan")

builder.add_edge("create_plan", "check_workload")

builder.add_conditional_edges(
    "check_workload",
    decide_next,
    {
        "correct": "correct_plan",
        "search": "search_resources"
    }
)

builder.add_edge("correct_plan", "check_workload")

builder.add_edge("search_resources", END)

In [ ]:
graph = builder.compile()

In [ ]:
result = graph.invoke({
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [],
    "status": "",
    "resources": {},
    "feedback": "",
    "weak_topics": []
})

In [ ]:
print(result["plan"])

[{'day': 1, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}, {'day': 2, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}, {'day': 3, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}]


In [ ]:
test_state = {
    "subjects": ["Python", "DBMS", "Machine Learning"],
    "hours": 3,
    "days": 3,
    "plan": [
        {
            "day": 1,
            "tasks": [
                {"subject": "Python", "hours": 1},
                {"subject": "DBMS", "hours": 1},
                {"subject": "Machine Learning", "hours": 1}
            ]
        }
    ],
    "status": "realistic",
    "resources": {},
    "feedback": "Machine Learning was difficult",
    "weak_topics": []
}

updated = analyze_feedback(test_state)
updated_plan = adjust_next_plan(test_state)

print(updated["plan"])

🔄 Plan adjusted using student feedback.
🔄 Next study plan adjusted!
[{'day': 1, 'tasks': [{'subject': 'Python', 'hours': 1}, {'subject': 'DBMS', 'hours': 1}, {'subject': 'Machine Learning', 'hours': 1.5}]}]


In [ ]:
def feedback_decision(state):
    if state["feedback"].strip():
        return "feedback"
    return "end"

In [ ]:
print(builder.nodes)

{'create_plan': StateNodeSpec(runnable=create_plan(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.StudyState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None, trace_policy=None), 'check_workload': StateNodeSpec(runnable=check_workload(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.StudyState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None, trace_policy=None), 'correct_plan': StateNodeSpec(runnable=correct_plan(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None, input_schema=<class '__main__.StudyState'>, retry_policy=None, cache_policy=None, is_error_handler=False, error_handler_node=None, ends=(), defer=False, timeout=None, trace_policy=None), 'search_resources': StateNodeSpec(runnable=sea

In [ ]:
builder.add_edge("analyze_feedback", "extract_weak_topics")

In [ ]:
builder.add_edge("extract_weak_topics", END)

In [ ]:
graph = builder.compile()

In [ ]:
result = graph.invoke({
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [],
    "status": "",
    "resources": {},
    "feedback": "",
    "weak_topics": []
})

In [ ]:
print(result["plan"])
print(result["resources"])

[{'day': 1, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}, {'day': 2, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}, {'day': 3, 'tasks': [{'subject': 'Python', 'hours': 1.0}, {'subject': 'DBMS', 'hours': 1.0}, {'subject': 'Machine Learning', 'hours': 1.0}]}]
{}


In [ ]:
from ddgs import DDGS

def search_weak_resources(state):
    resources = {}

    for subject in state["weak_topics"]:
        print(f"🔎 Searching resources for: {subject}")

        try:
            results = DDGS().text(
                f"{subject} beginner tutorial",
                max_results=3
            )

            resources[subject] = results

        except Exception as e:
            print(f"⚠️ Search failed for {subject}")
            print("Using empty resource list.")

            resources[subject] = []

    return {
        "resources": resources
    }

In [ ]:
builder.add_node("search_weak_resources", search_weak_resources)

In [ ]:
builder.add_edge("extract_weak_topics", END)

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(StudyState)

builder.add_node("create_plan", create_plan)
builder.add_node("check_workload", check_workload)
builder.add_node("correct_plan", correct_plan)
builder.add_node("search_resources", search_resources)
builder.add_node("analyze_feedback", analyze_feedback)
builder.add_node("extract_weak_topics", extract_weak_topics)
builder.add_node("search_weak_resources", search_weak_resources)

builder.add_edge(START, "create_plan")

builder.add_edge("create_plan", "check_workload")

builder.add_conditional_edges(
    "check_workload",
    decide_next,
    {
        "correct": "correct_plan",
        "search": "search_resources"
    }
)

builder.add_edge("correct_plan", "check_workload")

builder.add_edge("search_resources", END)

builder.add_edge("analyze_feedback", "extract_weak_topics")

builder.add_edge("extract_weak_topics", "search_weak_resources")

builder.add_edge("search_weak_resources", END)

graph = builder.compile()

print("✅ Graph compiled successfully!")

✅ Graph compiled successfully!


In [ ]:
test_state = {
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [
        {
            "day": 1,
            "tasks": [
                {"subject": "Python", "hours": 1},
                {"subject": "DBMS", "hours": 1},
                {"subject": "Machine Learning", "hours": 1}
            ]
        }
    ],
    "status": "realistic",
    "resources": {},
    "feedback": "Machine Learning was difficult and I could not complete DBMS",
    "weak_topics": []
}

result = graph.invoke(test_state)

print("Weak topics:", result["weak_topics"])

Weak topics: []


In [ ]:
test_state = {
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [],
    "status": "",
    "resources": {},
    "feedback": "Machine Learning was difficult and I could not complete DBMS",
    "weak_topics": []
}

result = extract_weak_topics(test_state)

print(result["weak_topics"])

📌 Weak topics: ['DBMS', 'Machine Learning']
['DBMS', 'Machine Learning']


In [ ]:
test_state["weak_topics"] = result["weak_topics"]

result = search_weak_resources(test_state)

print(result["resources"].keys())

🔎 Searching resources for weak topic: DBMS
🔎 Searching resources for weak topic: Machine Learning
dict_keys(['DBMS', 'Machine Learning'])


In [ ]:
result = search_weak_resources(test_state)

🔎 Searching resources for weak topic: DBMS
🔎 Searching resources for weak topic: Machine Learning


In [ ]:
print(result["resources"])

{'DBMS': [{'title': '10 Beginner SQL Practice Exercises With Solutions20 SQL Practice Problems for Beginners with SolutionsMySQL Exercises - W3SchoolsDBMS Tutorial - GeeksforGeeksSQL Exercises, Practice, Solution - w3resource', 'href': 'https://learnsql.com/blog/sql-practice-exercises/', 'body': 'The question is always where to find data for practicing SQL. We’ll use our dataset for all exercises. No need to limit yourself to this, though – you can find other free online datasets for practicing SQL. Our dataset consists of two tables. The table distribution_companieslists movie distribution companies with the following columns: 1. id– The I... See full list on learnsql.com 1. Select all data about distribution companies Solution: Solution explanation: Select the data using the SELECT statement. To select all the columns, use an asterisk (*). The table from which the data is selected is specified in the FROMclause. Solution output: 2. Select the title, IMDb rating, and release year for 

In [ ]:
for subject, resources in result["resources"].items():

    print("\n📚 Resources for:", subject)

    for i, resource in enumerate(resources, 1):
        print(f"\n{i}. {resource['title']}")
        print("🔗", resource['href'])
        print("📝", resource['body'])


📚 Resources for: DBMS

1. 10 Beginner SQL Practice Exercises With Solutions20 SQL Practice Problems for Beginners with SolutionsMySQL Exercises - W3SchoolsDBMS Tutorial - GeeksforGeeksSQL Exercises, Practice, Solution - w3resource
🔗 https://learnsql.com/blog/sql-practice-exercises/
📝 The question is always where to find data for practicing SQL. We’ll use our dataset for all exercises. No need to limit yourself to this, though – you can find other free online datasets for practicing SQL. Our dataset consists of two tables. The table distribution_companieslists movie distribution companies with the following columns: 1. id– The I... See full list on learnsql.com 1. Select all data about distribution companies Solution: Solution explanation: Select the data using the SELECT statement. To select all the columns, use an asterisk (*). The table from which the data is selected is specified in the FROMclause. Solution output: 2. Select the title, IMDb rating, and release year for each movie S

In [ ]:
def adjust_next_plan(state):
    new_plan = []

    for day in state["plan"]:
        tasks = []

        for task in day["tasks"]:

            subject = task["subject"]
            hours = task["hours"]

            if subject in state["weak_topics"]:
                hours += 0.5

            tasks.append({
                "subject": subject,
                "hours": hours
            })

        new_plan.append({
            "day": day["day"],
            "tasks": tasks
        })

    print("🔄 Next study plan adjusted!")

    return {
        "plan": new_plan
    }

In [ ]:
weak_result = extract_weak_topics(test_state)

print("Weak topics:", weak_result["weak_topics"])

📌 Weak topics: ['DBMS', 'Machine Learning']
Weak topics: ['DBMS', 'Machine Learning']


In [ ]:
resource_result = search_weak_resources(test_state)

In [ ]:
for subject, resources in resource_result["resources"].items():

    print("\n📚 Resources for:", subject)

    for i, resource in enumerate(resources, 1):
        print(f"{i}. {resource['title']}")
        print("🔗", resource["href"])
        print()

In [ ]:
def generate_final_result(state):

    # Find weak topics
    weak_result = extract_weak_topics(state)
    weak_topics = weak_result["weak_topics"]

    print("\n📌 WEAK TOPICS:", weak_topics)

    # Put weak topics into state
    state["weak_topics"] = weak_topics

    # Adjust study plan
    plan_result = adjust_next_plan(state)
    state["plan"] = plan_result["plan"]

    # Search resources
    resource_result = search_weak_resources(state)

    resources = resource_result["resources"]

    print("\n🔎 RESOURCES FOUND")

    for subject, results in resources.items():

        print(f"\n📚 {subject}")

        if not results:
            print("No resources found.")
            continue

        for i, item in enumerate(results, 1):
            print(f"\n{i}. {item['title']}")
            print(f"🔗 {item['href']}")
            print(f"📝 {item['body']}")

    # Save resources
    state["resources"] = resources

    return state

In [ ]:
final_result = generate_final_result(test_state)

📌 Weak topics: ['Machine Learning']
🔄 Next study plan adjusted!
🔎 Searching resources for weak topic: Machine Learning


In [ ]:
print("📌 WEAK TOPICS")
print(final_result["weak_topics"])

print("\n📅 UPDATED STUDY PLAN")
print(final_result["plan"])

📌 WEAK TOPICS
['Machine Learning']

📅 UPDATED STUDY PLAN
[{'day': 1, 'tasks': [{'subject': 'Python', 'hours': 1}, {'subject': 'DBMS', 'hours': 1}, {'subject': 'Machine Learning', 'hours': 1.5}]}]


In [ ]:
print("📌 WEAK TOPICS")
print(final_result["weak_topics"])

print("\n📅 UPDATED STUDY PLAN")
print(final_result["plan"])

📌 WEAK TOPICS
['Machine Learning']

📅 UPDATED STUDY PLAN
[{'day': 1, 'tasks': [{'subject': 'Python', 'hours': 1}, {'subject': 'DBMS', 'hours': 1}, {'subject': 'Machine Learning', 'hours': 1.5}]}]


In [ ]:
test_state = {
    "subjects": [
        "Python",
        "DBMS",
        "Machine Learning"
    ],
    "hours": 3,
    "days": 3,
    "plan": [
        {
            "day": 1,
            "tasks": [
                {"subject": "Python", "hours": 1},
                {"subject": "DBMS", "hours": 1},
                {"subject": "Machine Learning", "hours": 1}
            ]
        }
    ],
    "status": "realistic",
    "resources": {},
    "feedback": "Machine Learning was difficult and I could not complete DBMS",
    "weak_topics": []
}

In [ ]:
final_result = generate_final_result(test_state)

📌 Weak topics: ['DBMS', 'Machine Learning']

📌 WEAK TOPICS: ['DBMS', 'Machine Learning']
🔄 Next study plan adjusted!
🔎 Searching resources for: DBMS
🔎 Searching resources for: Machine Learning
⚠️ Search failed for Machine Learning
Using empty resource list.

🔎 RESOURCES FOUND

📚 DBMS

1. DBMS Tutorial - GeeksforGeeks
🔗 https://www.geeksforgeeks.org/dbms/dbms/
📝 Mar 14, 2026 · Understand the basics of DBMS, explore how it works, and learn about different database architectures used in real-world systems. It is a conceptual model for designing a databases to represent real-world data using entities, attributes, and relationships.

2. DBMS TutorialDBMS Tutorial: A Beginner’s Guide to Database Management ...Complete DBMS Tutorial: From Basics to AdvancedDBMS Tutorial - SimplilearnDBMS Tutorial for Beginners | Complete Database Management ...
🔗 https://www.tutorialspoint.com/dbms/index.htm
📝 Database Management System or DBMSin short refers to the technology of storing and retrieving users'

In [ ]:
!pip install -q requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

def search_web(query, max_results=3):

    url = "https://www.google.com/search?q=" + quote(query)

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=10)

    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    for result in soup.select("div.MjjYud")[:max_results]:

        title = result.select_one("h3")
        link = result.select_one("a")

        if title and link:
            results.append({
                "title": title.get_text(),
                "href": link.get("href"),
                "body": "Web resource for " + query
            })

    return results

In [ ]:
!pip uninstall -y ddgs
!pip install -U ddgs

Found existing installation: ddgs 9.16.0
Uninstalling ddgs-9.16.0:
  Successfully uninstalled ddgs-9.16.0
  Using cached ddgs-9.16.0-py3-none-any.whl.metadata (16 kB)
Using cached ddgs-9.16.0-py3-none-any.whl (47 kB)


In [ ]:
from ddgs import DDGS

results = DDGS().text(
    "DBMS beginner tutorial",
    max_results=5
)

print(results)

[{'title': 'DBMS Tutorial', 'href': 'https://www.tutorialspoint.com/dbms/index.htm', 'body': 'This tutorial explains the basics of DBMS such as its architecture, data models, data schema, data independence, E-R model, relation model, relational database design, and storage and file structure and much more.'}, {'title': 'DBMS Tutorial - GeeksforGeeks', 'href': 'https://www.geeksforgeeks.org/dbms/dbms/', 'body': 'A Database Management System (DBMS) is software used to manage data from a database. Published: March 14, 2026'}, {'title': 'DBMS Tutorial | Database Management System Tutorial for beginners | Tech Classes - YouTube', 'href': 'https://www.youtube.com/playlist?list=PLunlGNVWDAabuRMnz05LR5CFMeTjJffR3', 'body': 'In this tutorial, you will learn everything about database management systems (DBMS). #dbms #dbmstutorials #databasemanagementsystem'}, {'title': 'DBMS Tutorial: A Beginner’s Guide to Database Management Systems - DEV Community', 'href': 'https://dev.to/suraj_kumar_fb57ae09

In [ ]:
!pip install -q wikipedia

  Preparing metadata (setup.py) ... done


In [ ]:
import wikipedia

def search_web(query):
    try:
        results = wikipedia.search(query, results=5)

        useful_results = []

        for result in results:
            if "database" in result.lower() or "dbms" in result.lower():
                useful_results.append(result)

        return useful_results

    except Exception as e:
        print("Search error:", e)
        return []

In [ ]:
results = search_web("DBMS beginner tutorial")

print(results)



['Comparison of relational database management systems']


In [ ]:
results = search_web("Database management system")
print(results)

Search error: Expecting value: line 1 column 1 (char 0)
[]


In [ ]:
import requests

url = "https://en.wikipedia.org/wiki/Database"

response = requests.get(url)

print(response.status_code)

403


In [ ]:
import requests
from bs4 import BeautifulSoup

def search_web(query):
    url = "https://html.duckduckgo.com/html/"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        params={"q": query},
        headers=headers
    )

    print("Status:", response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    for result in soup.select(".result"):
        title = result.select_one(".result__title")
        link = result.select_one(".result__a")

        if title and link:
            results.append({
                "title": title.get_text(strip=True),
                "url": link.get("href")
            })

    return results[:5]

In [ ]:
results = search_web("DBMS beginner tutorial")

print(results)

Status: 202
[]


In [ ]:
print(results)

[]


In [ ]:
results = search_web("DBMS beginner tutorial")

print(results)

Status: 202
[]


In [ ]:
print(response.text[:1000])

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



In [ ]:
import requests
from bs4 import BeautifulSoup

def search_web(query):
    url = "https://html.duckduckgo.com/html/"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        params={"q": query},
        headers=headers
    )

    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    for result in soup.select(".result"):
        title = result.select_one(".result__title")
        link = result.select_one(".result__a")

        if title and link:
            results.append({
                "title": title.get_text(strip=True),
                "url": link.get("href")
            })

    return results[:5]

In [ ]:
results = search_web("DBMS beginner tutorial")

for r in results:
    print("Title:", r["title"])
    print("URL:", r["url"])
    print()

Status: 202


In [ ]:
results = search_web("DBMS beginner tutorial")

print("Number of results:", len(results))
print(results)

Status: 202
Number of results: 0
[]


In [ ]:
print(response.text[:2000])

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



In [ ]:
def search_web(query):

    courses = {
        "DBMS": [
            {
                "course": "DBMS Tutorial",
                "resource": "https://www.geeksforgeeks.org/dbms/"
            },
            {
                "course": "SQL Tutorial",
                "resource": "https://www.w3schools.com/sql/"
            },
            {
                "course": "Database Management Systems",
                "resource": "https://www.javatpoint.com/dbms-tutorial"
            }
        ]
    }

    return courses["DBMS"]

In [ ]:
results = search_web("DBMS beginner tutorial")

for result in results:
    print("Course:", result["course"])
    print("Resource:", result["resource"])
    print()

Course: DBMS Tutorial
Resource: https://www.geeksforgeeks.org/dbms/

Course: SQL Tutorial
Resource: https://www.w3schools.com/sql/

Course: Database Management Systems
Resource: https://www.javatpoint.com/dbms-tutorial



In [ ]:
def search_web(query):

    courses = {
        "DBMS": [
            ("DBMS Tutorial", "https://www.geeksforgeeks.org/dbms/"),
            ("SQL Tutorial", "https://www.w3schools.com/sql/"),
            ("DBMS Course", "https://www.javatpoint.com/dbms-tutorial")
        ],

        "Python": [
            ("Python Tutorial", "https://www.w3schools.com/python/"),
            ("Python Course", "https://www.geeksforgeeks.org/python/python-programming-language-tutorial/"),
            ("Python Documentation", "https://docs.python.org/3/tutorial/")
        ],

        "Machine Learning": [
            ("Machine Learning Tutorial", "https://www.geeksforgeeks.org/machine-learning/machine-learning/"),
            ("Machine Learning Course", "https://www.coursera.org/learn/machine-learning"),
            ("Machine Learning Crash Course", "https://developers.google.com/machine-learning/crash-course")
        ],

        "Artificial Intelligence": [
            ("AI Tutorial", "https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence/"),
            ("AI Course", "https://www.coursera.org/browse/computer-science/artificial-intelligence"),
            ("AI Learning", "https://www.ibm.com/think/topics/artificial-intelligence")
        ],

        "Data Science": [
            ("Data Science Tutorial", "https://www.geeksforgeeks.org/data-science/data-science-for-beginners/"),
            ("Data Science Course", "https://www.coursera.org/browse/data-science"),
            ("Data Science Tutorial", "https://www.w3schools.com/datascience/")
        ],

        "NumPy": [
            ("NumPy Tutorial", "https://numpy.org/learn/"),
            ("NumPy Tutorial", "https://www.w3schools.com/python/numpy/"),
            ("NumPy Documentation", "https://numpy.org/doc/stable/user/quickstart.html")
        ],

        "TensorFlow": [
            ("TensorFlow Tutorials", "https://www.tensorflow.org/tutorials"),
            ("TensorFlow Beginner Course", "https://www.tensorflow.org/learn"),
            ("TensorFlow Documentation", "https://www.tensorflow.org/api_docs")
        ],

        "SQL": [
            ("SQL Tutorial", "https://www.w3schools.com/sql/"),
            ("SQL Tutorial", "https://www.geeksforgeeks.org/sql/sql-tutorial/"),
            ("SQL Course", "https://www.coursera.org/learn/sql-for-data-science")
        ],

        "Java": [
            ("Java Tutorial", "https://www.w3schools.com/java/"),
            ("Java Tutorial", "https://www.geeksforgeeks.org/java/java/"),
            ("Java Documentation", "https://docs.oracle.com/en/java/")
        ],

        "Git": [
            ("Git Tutorial", "https://git-scm.com/docs/gittutorial"),
            ("Git Tutorial", "https://www.w3schools.com/git/"),
            ("GitHub Learning", "https://skills.github.com/")
        ]
    }

    for course_name in courses:
        if course_name.lower() in query.lower():
            return courses[course_name]

    return []

In [ ]:
results = search_web("Python")

for course, resource in results:
    print("Course:", course)
    print("Resource:", resource)
    print()

Course: Python Tutorial
Resource: https://www.w3schools.com/python/

Course: Python Course
Resource: https://www.geeksforgeeks.org/python/python-programming-language-tutorial/

Course: Python Documentation
Resource: https://docs.python.org/3/tutorial/



In [ ]:
results = search_web("Machine Learning")

for course, resource in results:
    print("Course:", course)
    print("Resource:", resource)

Course: Machine Learning Tutorial
Resource: https://www.geeksforgeeks.org/machine-learning/machine-learning/
Course: Machine Learning Course
Resource: https://www.coursera.org/learn/machine-learning
Course: Machine Learning Crash Course
Resource: https://developers.google.com/machine-learning/crash-course


In [ ]:
!pip install -q tavily-python google-search-results


  Preparing metadata (setup.py) ... done


In [ ]:
!pip install -q tavily-python requests

In [ ]:
from tavily import TavilyClient
import requests

tavily = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

def tavily_search(query):
    results = tavily.search(
        query=query,
        max_results=5
    )

    return [
        {
            "course": r["title"],
            "resource": r["url"]
        }
        for r in results["results"]
    ]

In [ ]:
def serper_search(query):
    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": os.environ["SERPER_API_KEY"],
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json={"q": query}
    )

    data = response.json()

    results = []

    for r in data.get("organic", [])[:5]:
        results.append({
            "course": r["title"],
            "resource": r["link"]
        })

    return results

In [ ]:
def search_web(query):

    try:
        results = tavily_search(query)

        if results:
            return results

    except Exception as e:
        print("Tavily failed. Trying Serper...")

    try:
        return serper_search(query)

    except Exception as e:
        print("Serper failed:", e)
        return []

In [ ]:

results = search_web("DBMS beginner course")

for r in results:
    print("Course:", r["course"])
    print("Resource:", r["resource"])
    print()

Course: DBMS Full Course for Beginners | Learn Database Management System from Scratch | What is DBMS
Resource: https://www.youtube.com/watch?v=c5HAwKX-suM

Course: Free DBMS Online Course with Certificate 2026
Resource: https://www.scaler.com/topics/course/dbms

Course: Courses for Database Core Concepts Beginner - Skillsoft
Resource: https://www.skillsoft.com/channel/database-core-concepts-c5d9e950-e198-11e6-91a7-0242c0a80704?expertiselevel=95551

Course: Best Database Courses & Certificates [2026] | Coursera
Resource: https://www.coursera.org/courses?query=database

Course: DBMS Tutorial
Resource: https://www.tutorialspoint.com/dbms/index.htm



In [ ]:
!pip install -q langsmith

In [ ]:
from langsmith import traceable

@traceable(name="Web Search")
def search_web(query):

    try:
        results = tavily_search(query)

        if results:
            return results

    except Exception:
        print("Tavily failed. Trying Serper...")

    try:
        return serper_search(query)

    except Exception as e:
        print("Serper failed:", e)
        return []

In [ ]:
results = search_web("DBMS beginner course")

for r in results:
    print("Course:", r["course"])
    print("Resource:", r["resource"])

Course: Free Database Management System Course With Certificate For Beginners
Resource: https://www.mygreatlearning.com/academy/learn-for-free/courses/database-management-system
Course: Free DBMS Online Course with Certificate 2026
Resource: https://www.scaler.com/topics/course/dbms
Course: Learn Database Management System from Scratch
Resource: https://www.youtube.com/watch?v=c5HAwKX-suM
Course: DBMS and MySQL Complete Course Beginner to Advance
Resource: https://www.udemy.com/course/dbms-and-mysql-for-big-data-and-data-science
Course: Courses for Database Core Concepts Beginner
Resource: https://www.skillsoft.com/channel/database-core-concepts-c5d9e950-e198-11e6-91a7-0242c0a80704?expertiselevel=95551


In [ ]:
from langsmith import traceable

@traceable(name="AI Project Mentor")
def project_mentor(domain, skill_level):

    query = f"{domain} {skill_level} beginner course"

    resources = search_web(query)

    return {
        "domain": domain,
        "skill_level": skill_level,
        "resources": resources
    }

In [ ]:
result = project_mentor(
    "DBMS",
    "Beginner"
)

print(result)

{'domain': 'DBMS', 'skill_level': 'Beginner', 'resources': [{'course': 'Courses for Database Core Concepts Beginner', 'resource': 'https://www.skillsoft.com/channel/database-core-concepts-c5d9e950-e198-11e6-91a7-0242c0a80704?expertiselevel=95551'}, {'course': 'Free Course on Database Management System', 'resource': 'https://www.mygreatlearning.com/academy/learn-for-free/courses/database-management-system'}, {'course': 'DBMS and MySQL Complete Course Beginner to Advance', 'resource': 'https://www.udemy.com/course/dbms-and-mysql-for-big-data-and-data-science'}, {'course': 'Learn Database Management System from Scratch', 'resource': 'https://www.youtube.com/watch?v=c5HAwKX-suM'}, {'course': 'Free DBMS Online Course with Certificate 2026', 'resource': 'https://www.scaler.com/topics/course/dbms'}]}


In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os

print(os.listdir("/content"))

['.config', 'app.py', 'log.txt', 'client_secret_364859831951-q7e7rpi001nsbjlvrhq3jncac5th55c8.apps.googleusercontent.com.json', 'sample_data']


In [ ]:
import os

for file in os.listdir("/content"):
    print(file)

.config
app.py
log.txt
client_secret_364859831951-q7e7rpi001nsbjlvrhq3jncac5th55c8.apps.googleusercontent.com.json
sample_data


In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

flow = InstalledAppFlow.from_client_secrets_file(
    filename,
    SCOPES
)

auth_url, _ = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print(auth_url)

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=364859831951-q7e7rpi001nsbjlvrhq3jncac5th55c8.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=FiQoOj9eHK3R6Ij3I4HcY1AsmBlHK9&code_challenge=8ED3otqKycfb8aNMVRjkmV3DW-J3jXZ3ToCWCLV2Ix8&code_challenge_method=S256&access_type=offline&prompt=consent


In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib pyngrok

In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google.colab import output

url = output.eval_js("""
(async () => {
  return await google.colab.kernel.proxyPort(8080);
})()
""")

print(url)

https://8080-m-s-kkb-ase1a0-1di965kuk89e0-a.asia-east1-0.prod.colab.dev


In [ ]:
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse
import threading

PORT = 8081

callback_data = {}
callback_event = threading.Event()

class OAuthHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)

        if parsed.path == "/oauth2callback":
            callback_data["query"] = parsed.query

            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()

            self.wfile.write(b"""
            <html>
            <body>
                <h2>Gmail authentication successful!</h2>
                <p>You can close this tab and return to Colab.</p>
            </body>
            </html>
            """)

            callback_event.set()

    def log_message(self, format, *args):
        pass

server = HTTPServer(("0.0.0.0", PORT), OAuthHandler)

thread = threading.Thread(
    target=server.serve_forever,
    daemon=True
)
thread.start()

from google.colab import output

proxy_url = output.eval_js(
    f"google.colab.kernel.proxyPort({PORT})"
)

redirect_uri = proxy_url.rstrip("/") + "/oauth2callback"

print("REDIRECT URI:")
print(redirect_uri)

REDIRECT URI:
https://8081-m-s-kkb-ase1a0-1di965kuk89e0-a.asia-east1-0.prod.colab.dev/oauth2callback


In [ ]:
from getpass import getpass

CLIENT_ID = getpass("Enter Client ID: ")
CLIENT_SECRET = getpass("Enter Client Secret: ")
REFRESH_TOKEN = getpass("Enter Refresh Token: ")

Enter Client ID: ··········
Enter Client Secret: ··········
Enter Refresh Token: ··········


In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("Paste your Access Token: ")

Paste your Access Token: ··········


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(token=ACCESS_TOKEN)

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Gmail API connected!")

✅ Gmail API connected!


In [ ]:
results = service.users().messages().list(
    userId="me",
    maxResults=5
).execute()

messages = results.get("messages", [])

print("Messages found:", len(messages))

Messages found: 5


In [ ]:
results = service.users().messages().list(
    userId="me",
    maxResults=10
).execute()

messages = results.get("messages", [])

print("Emails found:", len(messages))

Emails found: 10


In [ ]:
for msg in messages:
    email = service.users().messages().get(
        userId="me",
        id=msg["id"],
        format="metadata",
        metadataHeaders=["Subject", "From"]
    ).execute()

    headers = email["payload"]["headers"]

    subject = ""
    sender = ""

    for header in headers:
        if header["name"] == "Subject":
            subject = header["value"]
        elif header["name"] == "From":
            sender = header["value"]

    print("From:", sender)
    print("Subject:", subject)
    print("-" * 50)

From: LinkedIn Job Alerts <jobalerts-noreply@linkedin.com>
Subject: Asian Paints is hiring a EXECUTIVE N - WOOD FINISHES RELATIONSHIP (Calicut)
--------------------------------------------------
From: Google <no-reply@accounts.google.com>
Subject: 2-Step Verification turned on
--------------------------------------------------
From: LangChain <hello@mail.langchain.com>
Subject: Welcome to LangSmith!
--------------------------------------------------
From: Google <noreply-accounts@google.com>
Subject: You shared some Google Account data with LangSmith
--------------------------------------------------
From: Serper <support@serper.dev>
Subject: Reset Your Serper Password
--------------------------------------------------
From: Serper <support@serper.dev>
Subject: Reset Your Serper Password
--------------------------------------------------
From: Serper <support@serper.dev>
Subject: Please verify your email address
--------------------------------------------------
From: Google <noreply-a

In [ ]:
results = service.users().messages().list(
    userId="me",
    q="(exam OR assignment OR deadline OR test OR project)",
    maxResults=10
).execute()

messages = results.get("messages", [])

print("Study-related emails:", len(messages))

Study-related emails: 10


In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("Paste your Gmail access token: ")

Paste your Gmail access token: ··········


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(token=ACCESS_TOKEN)

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Gmail sending enabled!")

✅ Gmail sending enabled!


In [ ]:
MY_EMAIL = "jeevithajeevitha3420@gmail.com"

In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("Enter NEW Access Token: ")

Enter NEW Access Token: ··········


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(
    token=ACCESS_TOKEN,
    scopes=["https://www.googleapis.com/auth/gmail.send"]
)

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Gmail send permission loaded!")

✅ Gmail send permission loaded!


In [ ]:
import requests

info = requests.get(
    "https://oauth2.googleapis.com/tokeninfo",
    params={"access_token": ACCESS_TOKEN}
).json()

print("Scopes granted:")
print(info.get("scope"))

Scopes granted:
https://www.googleapis.com/auth/gmail.readonly


In [ ]:
import requests

info = requests.get(
    "https://oauth2.googleapis.com/tokeninfo",
    params={"access_token": ACCESS_TOKEN}
).json()

print("Scopes granted:")
print(info.get("scope"))

Scopes granted:
https://www.googleapis.com/auth/gmail.readonly


In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("Enter NEW Access Token: ")

Enter NEW Access Token: ··········


In [ ]:
import requests

info = requests.get(
    "https://oauth2.googleapis.com/tokeninfo",
    params={"access_token": ACCESS_TOKEN}
).json()

print("Scopes granted:")
print(info.get("scope"))

Scopes granted:
https://www.googleapis.com/auth/gmail.readonly


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(token=ACCESS_TOKEN)

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Gmail send access ready")

✅ Gmail send access ready


In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("Enter NEW Access Token: ")

Enter NEW Access Token: ··········


In [ ]:
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(token=ACCESS_TOKEN)

service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Gmail connected!")

✅ Gmail connected!


In [ ]:
import requests

check = requests.get(
    "https://oauth2.googleapis.com/tokeninfo",
    params={"access_token": ACCESS_TOKEN}
)

print(check.json())

{'azp': '407408718192.apps.googleusercontent.com', 'aud': '407408718192.apps.googleusercontent.com', 'scope': 'https://www.googleapis.com/auth/gmail.readonly', 'exp': '1790235335', 'expires_in': '2189', 'access_type': 'offline'}


In [ ]:
!pip install -q google-api-python-client

In [ ]:
Credentials(token=ACCESS_TOKEN)

In [ ]:
import smtplib
from email.message import EmailMessage
from getpass import getpass


In [ ]:
email = "jeevithajeevitha3420@gmail.com"
app_password = getpass("Enter your Gmail App Password: ")

Enter your Gmail App Password: ··········


In [ ]:
from getpass import getpass

email = "jeevithajeevitha3420@gmail.com"
app_password = getpass("Enter your App Password: ")

Enter your App Password: ··········


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import smtplib
from email.message import EmailMessage

msg = EmailMessage()

msg["From"] = email
msg["To"] = email
msg["Subject"] = "Self-Correcting Study Planner Test"

msg.set_content(
    """Hello!

This is a test email from my Self-Correcting Study Planner.

Gmail email sending is working successfully!
"""
)

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
    smtp.login(email, app_password)
    smtp.send_message(msg)

print("✅ Email sent successfully!")

✅ Email sent successfully!
